### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="asp_potassco_classification",
    dataset_year="2014",
    domain_str="technology & internet",
    # Data Source
    dataset_source="ASlib",
    original_dataset_source_download_link="https://github.com/coseal/aslib_data/tree/master/ASP-POTASSCO",
    download_description="""
wget https://raw.githubusercontent.com/coseal/aslib_data/refs/heads/master/ASP-POTASSCO/algorithm_runs.arff \
&& wget https://raw.githubusercontent.com/coseal/aslib_data/refs/heads/master/ASP-POTASSCO/feature_values.arff \
&& mkdir -p local-data-warehouse/asp_potassco_classification \
&& mv feature_values.arff algorithm_runs.arff local-data-warehouse/asp_potassco_classification/
""",
    # References
    academic_reference_bibtex="""@article{hoos2014claspfolio,
  title={claspfolio 2: Advances in algorithm selection for answer set programming},
  author={Hoos, Holger and Lindauer, Marius and Schaub, Torsten},
  journal={Theory and Practice of Logic Programming},
  volume={14},
  number={4-5},
  pages={569--585},
  year={2014},
  publisher={Cambridge University Press}
}
@article{bischl_aslib_2016,
	title = {{ASlib}: {A} {Benchmark} {Library} for {Algorithm} {Selection}},
	number = {237},
	journal = {Artificial Intelligence Journal (AIJ)},
	author = {Bischl, Bernd and Kerschke, Pascal and Kotthoff, Lars and Lindauer, Marius and Malitsky, Yuri and Fréchette, Alexandre and Hoos, Holger H. and Hutter, Frank and Leyton-Brown, Kevin and Tierney, Kevin and Vanschoren, Joaquin},
	year = {2016},
	pages = {41--58}
}
""",
    academic_reference_bibtex_key="hoos2014claspfolio,bischl_aslib_2016",
    license="GPLv3",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
We get the data from ASlib and merge them into one file.

- We treat it as a multiclass classification task to solve the algorithm selection task as in the OpenML version (https://openml.org/d/41705).
- We drop all cases where no algorithm was able to finish before the timeout as these are essentially random labels.
- We resole the instance ID to a task ID by mapping the instance ID back to the source task based on their naming convention. From our understanding, the data contains multiple instance from the same solver task, differing only in seed or task configurations. We want to avoid having samples from the same task in both train and test set, as this would leak what algorithms are best for this task, and since in real-world you might not have samples from your new task. Thus, we treat the data as a grouped task, where we aim to generalize the predicting the best algorithm to new tasks across a set of instances of this task.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="algorithm",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss", # or RMSE on runtime, or PAR10?
    # For classification
    stratify_on="algorithm",
    # For grouped data
    group_on="task_id",
    group_labels="per_sample",
)

## Preprocessing

In [2]:
import arff
import pandas as pd
import uuid
import numpy as np


def load_arff(path) -> pd.DataFrame:
    with open(path, encoding="utf-8") as f:
        data = arff.load(f)
    df = pd.DataFrame(data["data"], columns=[a[0] for a in data["attributes"]])
    return df


df_features = load_arff(dataset_mold.path / "feature_values.arff").drop(columns=["repetition"])
df_algo_runs = load_arff(dataset_mold.path / "algorithm_runs.arff")

# Only keep instances for which we have meaningful labels
df_algo_runs = df_algo_runs[df_algo_runs["runstatus"] == "ok"]
# Reduce to best algorithm per instance
df_algo_runs = df_algo_runs.loc[
    df_algo_runs.groupby('instance_id')['runtime'].idxmin(),
    ['instance_id', 'algorithm']
].reset_index(drop=True)

# Merge
df = df_algo_runs.merge(df_features, on="instance_id", how="left")
print("Merged data shape:", df.shape)


# Map instance ID to group ID
df["task_id"]  = df["instance_id"].str.rsplit("/", n=1).str[0]
# map it to a uuid
mapping = {val: uuid.uuid4().hex[:12] for val in df["task_id"].unique()}
df["task_id"] = df["task_id"].map(mapping)
df = df.drop(columns=[
    "instance_id",
    # Duplicated columns
    "Frac_Removed_Nogood-1",
    "Frac_Removed_Nogood-2",
])

as_cat_type = ["task_id", "algorithm"]
df[as_cat_type] = df[as_cat_type].astype("category")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Merged data shape: (1212, 140)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 1,212
Columns: 138
Use sampling: False (sample size: 1,212)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Constraints', 'Binary_Constraints', 'Problem_Variables', 'Equivalences', 'Other_Equivalences', 'Created_Bodies', 'Normal_Rules', 'Rules', 'Free_Problem_Variables', 'Avg_Conflict_Levels-2']
Rows remaining as candidates after top-10 filter: 62 (of 1,212)

#### Duplicate Report
Total duplicate rows: 5 (0.41% of dataset)
Duplicate rows ignoring target: 10 (0.83% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,algorithm,Frac_Neg_Body,Frac_Pos_Body,Frac_Unary_Rules,Frac_Binary_Rules,Frac_Ternary_Rules,Frac_Integrity_Rules,Tight,Problem_Variables,Free_Problem_Variables,Assigned_Problem_Variables,Constraints,Constraints/Vars,Created_Bodies,Program_Atoms,SCCS,Nodes_in_Positive_BADG,Rules,Normal_Rules,Cardinality_Rules,Choice_Rules,Weight_Rules,Frac_Normal_Rules,Frac_Cardinality_Rules,Frac_Choice_Rules,Frac_Weight_Rules,Equivalences,Atom-Atom_Equivalences,Body-Body_Equivalences,Other_Equivalences,Frac_Atom-Atom_Equivalences,Frac_Body-Body_Equivalences,Frac_Other_Equivalences,Binary_Constraints,Ternary_Constraints,Other_Constraints,Frac_Binary_Constraints,Frac_Ternary_Constraints,Frac_Other_Constraints,Choices-1,Conflicts/Choices-1,Avg_Conflict_Levels-1,Avg_LBD_Levels-1,Learnt_from_Conflict-1,Learnt_from_Loop-1,Frac_Learnt_from_Conflict-1,Frac_Learnt_from_Loop-1,Literals_in_Conflict_Nogoods-1,Literals_in_Loop_Nogoods-1,Frac_Literals_in_Conflict_Nogoods-1,Frac_Literals_in_Loop_Nogoods-1,Removed_Nogoods-1,Learnt_Binary-1,Learnt_Ternary-1,Learnt_Others-1,Frac_Learnt_Binary-1,Frac_Learnt_Ternary-1,Frac_Learnt_Others-1,Skipped_Levels_while_Backjumping-1,Avg_Skipped_Levels_while_Backjumping-1,Longest_Backjumping-1,Running_Avg_Conflictlevel-1,Running_Avg_LBD-1,Choices-2,Conflicts/Choices-2,Avg_Conflict_Levels-2,Avg_LBD_Levels-2,Learnt_from_Conflict-2,Learnt_from_Loop-2,Frac_Learnt_from_Conflict-2,Frac_Learnt_from_Loop-2,Literals_in_Conflict_Nogoods-2,Literals_in_Loop_Nogoods-2,Frac_Literals_in_Conflict_Nogoods-2,Frac_Literals_in_Loop_Nogoods-2,Removed_Nogoods-2,Learnt_Binary-2,Learnt_Ternary-2,Learnt_Others-2,Frac_Learnt_Binary-2,Frac_Learnt_Ternary-2,Frac_Learnt_Others-2,Skipped_Levels_while_Backjumping-2,Avg_Skipped_Levels_while_Backjumping-2,Longest_Backjumping-2,Running_Avg_Conflictlevel-2,Running_Avg_LBD-2,Choices-3,Conflicts/Choices-3,Avg_Conflict_Levels-3,Avg_LBD_Levels-3,Learnt_from_Conflict-3,Learnt_from_Loop-3,Frac_Learnt_from_Conflict-3,Frac_Learnt_from_Loop-3,Literals_in_Conflict_Nogoods-3,Literals_in_Loop_Nogoods-3,Frac_Literals_in_Conflict_Nogoods-3,Frac_Literals_in_Loop_Nogoods-3,Removed_Nogoods-3,Learnt_Binary-3,Learnt_Ternary-3,Learnt_Others-3,Frac_Removed_Nogood-3,Frac_Learnt_Binary-3,Frac_Learnt_Ternary-3,Frac_Learnt_Others-3,Skipped_Levels_while_Backjumping-3,Avg_Skipped_Levels_while_Backjumping-3,Longest_Backjumping-3,Running_Avg_Conflictlevel-3,Running_Avg_LBD-3,Choices-4,Conflicts/Choices-4,Avg_Conflict_Levels-4,Avg_LBD_Levels-4,Learnt_from_Conflict-4,Learnt_from_Loop-4,Frac_Learnt_from_Conflict-4,Frac_Learnt_from_Loop-4,Literals_in_Conflict_Nogoods-4,Literals_in_Loop_Nogoods-4,Frac_Literals_in_Conflict_Nogoods-4,Frac_Literals_in_Loop_Nogoods-4,Removed_Nogoods-4,Learnt_Binary-4,Learnt_Ternary-4,Learnt_Others-4,Frac_Removed_Nogood-4,Frac_Learnt_Binary-4,Frac_Learnt_Ternary-4,Frac_Learnt_Others-4,Skipped_Levels_while_Backjumping-4,Avg_Skipped_Levels_while_Backjumping-4,Longest_Backjumping-4,Running_Avg_Conflictlevel-4,Running_Avg_LBD-4,task_id
0,clasp/2.1.3/h11-n1,0.0094,0.8846,0.0077,0.0045,0.5951,0.4004,0.0,53071.0,44709.0,8362.0,158491.0,2.9864,70097.0,36685.0,100.0,45217.0,71117.0,71086.0,16.0,15.0,0.0,0.9996,0.0002,0.0002,0.0,5763.0,850.0,1071.0,3842.0,0.1475,0.1858,0.6667,106634.0,42268.0,9589.0,0.6728,0.2667,0.0605,84.0,0.3929,11.7879,2.9091,33.0,1338.0,0.0241,0.9759,188.0,42154.0,0.0044,0.9956,0.0,19.0,2.0,1350.0,0.0139,0.0015,0.9847,71.0,2.1515,8.0,3.89,0.96,224.0,0.2902,12.4308,3.1538,65.0,2599.0,0.0244,0.9756,452.0,73881.0,0.0061,0.9939,0.0,27.0,8.0,2629.0,0.0101,0.0030,0.9869,197.0,3.0308,19.0,8.08,2.05,367.0,0.2643,11.4227,3.0206,97.0,3670.0,0.0257,0.9743,611.0,100565.0,0.0060,0.9940,0.0,37.0,14.0,3716.0,0.0,0.0098,0.0037,0.9865,337.0,3.4742,19.0,11.08,2.93,428.0,0.3014,9.6977,2.8450,129.0,4119.0,0.0304,0.9696,774.0,111704.0,0.0069,0.9931,0.0,51.0,17.0,4180.0,0.0,0.0120,0.0040,0.9840,396.0,3.0698,19.0,9.34,2.79,2134e8d16e0d
1,clasp/2.1.3/h4-n1,0.0073,0.7807,0.0051,0.6897,0.2181,0.1476,1.0,9283.0,6338.0,2945.0,2

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,algorithm,category,0.0,0.00,11.0,"clasp/2.1.3/h1-n1, clasp/2.1.3/h10-n1, clasp/2.1.3/h6-n1, clasp/2.1.3/h4-n1, clasp/2.1.3/h8-n1, clasp/2.1.3/h7-n1, clasp/2.1.3/h2-n1, clasp/2.1.3/h11-n1, clasp/2.1.3/h9-n1, clasp/2.1.3/h5-n1"
1,task_id,category,0.0,0.00,96.0,"81728f2574c3, 8f6a79c51a4b, 00cf7b166f3c, e57bbf07e3ce, fd0e9d0851f7, 2b9a735d9c1d, 12fc10127cde, 27c3fb2ec1c4, 4806e1713598, cf4b46f8eb3c"
2,Choices-4,float64,218.0,17.99,751.0,"528.0, 200.0, 644.0, 716.0, 634.0, 974.0, 415.0, 557.0, 128.0, 411.0"
3,Conflicts/Choices-4,float64,218.0,17.99,812.0,"0.2424, 0.2524, 0.1427, 0.2393, 0.0047, 0.0048, 0.0075, 0.2622, 0.2298, 0.251"
4,Avg_Conflict_Levels-4,float64,218.0,17.99,954.0,"33.8984, 64.6589, 1.0, 38.1016, 33.1797, 10.3125, 19.8438, 49.6692, 45.3594, 53.3566"
5,Avg_LBD_Levels-4,float64,218.0,17.99,825.0,"3.1406, 4.0231, 10.6641, 3.5469, 3.9922, 2.8281, 4.1008, 3.0, 5.4297, 3.2578"
6,Learnt_from_Conflict-4,float64,218.0,17.99,10.0,"128.0, 129.0, 130.0, 131.0, 132.0, 133.0, 134.0, 135.0, 136.0, 120.0"
7,Learnt_from_Loop-4,float64,218.0,17.99,284.0,"0.0, 28.0, 31.0, 30.0, 37.0, 11.0, 9.0, 32.0, 18.0, 26.0"
8,Frac_Learnt_from_Conflict-4,float64,218.0,17.99,312.0,"1.0, 0.8205, 0.805, 0.8101, 0.9209, 0.9343, 0.8312, 0.8767, 0.8421, 0.9481"
9,Frac_Learnt_from_Loop-4,float64,218.0,17.99,312.0,"0.0, 0.1795, 0.195, 0.1899, 0.0791, 0.0657, 0.1688, 0.1233, 0.1579, 0.0519"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Frac_Neg_Body,1196.0,0.133759,2.194506e-01,0.0002,9.283000e-01
Frac_Pos_Body,1196.0,0.790592,2.101437e-01,0.0000,9.994000e-01
Frac_Unary_Rules,1196.0,0.103597,2.008177e-01,0.0000,8.863000e-01
Frac_Binary_Rules,1196.0,0.191836,1.667393e-01,0.0000,9.737000e-01
Frac_Ternary_Rules,1196.0,0.494413,2.773479e-01,0.0000,1.000000e+00
Frac_Integrity_Rules,1196.0,0.282627,2.775921e-01,0.0000,9.977000e-01
Tight,1196.0,0.517559,4.999006e-01,0.0000,1.000000e+00
Problem_Variables,1196.0,94270.510033,3.614105e+05,59.0000,7.758660e+06
Free_Problem_Variables,1196.0,50633.986622,1.015189e+05,45.0000,1.691690e+06
Assigned_Problem_Variables,1196.0,43636.523411,3.151337e+05,2.0000,6.910879e+06


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column    rank                                  
algorithm 1      clasp/2.1.3/h1-n1    240  19.80
          2     clasp/2.1.3/h10-n1    176  14.52
          3      clasp/2.1.3/h6-n1    151  12.46
          4      clasp/2.1.3/h4-n1    125  10.31
          5      clasp/2.1.3/h8-n1    109   8.99
task_id   1           81728f2574c3    134  11.06
          2           8f6a79c51a4b     89   7.34
          3           00cf7b166f3c     74   6.11
          4           e57bbf07e3ce     67   5.53
          5           fd0e9d0851f7     59   4.87

In [8]:
# Target Distribution
target_df

,count,pct
algorithm,,
clasp/2.1.3/h1-n1,240,19.80
clasp/2.1.3/h10-n1,176,14.52
clasp/2.1.3/h6-n1,151,12.46
clasp/2.1.3/h4-n1,125,10.31
clasp/2.1.3/h8-n1,109,8.99
clasp/2.1.3/h7-n1,88,7.26
clasp/2.1.3/h2-n1,87,7.18
clasp/2.1.3/h11-n1,75,6.19
clasp/2.1.3/h9-n1,75,6.19


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For Grouped Non-IID data
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
)


splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified Grouped splits.
Using label-per-sample grouped splits.


Repeat 0, Fold 0:
            Train N: 733, Test N: 479
            Target Distribution:
            	Train target distribution: {'clasp/2.1.3/h1-n1': 0.28512960436562074, 'clasp/2.1.3/h4-n1': 0.1296043656207367, 'clasp/2.1.3/h10-n1': 0.10641200545702592, 'clasp/2.1.3/h2-n1': 0.08594815825375171, 'clasp/2.1.3/h6-n1': 0.08321964529331514, 'clasp/2.1.3/h7-n1': 0.0791268758526603, 'clasp/2.1.3/h11-n1': 0.06957708049113233, 'clasp/2.1.3/h5-n1': 0.058663028649386086, 'clasp/2.1.3/h8-n1': 0.047748976807639835, 'clasp/2.1.3/h9-n1': 0.04229195088676671, 'clasp/2.1.3/h3-n1': 0.01227830832196453}
            	Test target distribution: {'clasp/2.1.3/h10-n1': 0.2045929018789144, 'clasp/2.1.3/h6-n1': 0.18789144050104384, 'clasp/2.1.3/h8-n1': 0.1544885177453027, 'clasp/2.1.3/h9-n1': 0.0918580375782881, 'clasp/2.1.3/h1-n1': 0.06471816283924843, 'clasp/2.1.3/h7-n1': 0.06263048016701461, 'clasp/2.1.3/h4-n1': 0.06263048016701461, 'clasp/2.1.3/h11-n1': 0.05010438413361169, 'clasp/2.1.3/h2-n1': 0.05010438

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to asp_potassco_classification/019d73d2-fce7-7f07-82e1-4882cb552f37


019d73d2-fce7-7f07-82e1-4882cb552f37
cd9153b1d99e94e64e1e12fb527a8093abe5128feb79eede723de6e4b3b1694f
